In [137]:
pip install textblob

Note: you may need to restart the kernel to use updated packages.


In [138]:
import pandas as pd
import numpy as np
from textblob import TextBlob
from sklearn.preprocessing import MinMaxScaler

In [139]:
product_df = pd.read_csv("product_level_socred.csv")
review_df = pd.read_csv("review_level_table.csv")

print("Product preview:")
display(product_df.head())

print("Review Preview:")
display(review_df.head())
      

Product preview:


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,img_link,...,log_rating_count,price_gap,discount_ratio_calc,review_strength,discount_efficiency_score,rating_norm,log_rating_count_norm,discount_norm,product_performance_score,product_segment
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,399.0,1099.0,0.64,4.2,24269.0,High Compatibility : Compatible With iPhone 12...,https://m.media-amazon.com/images/W/WEBP_40237...,...,10.10,700.0,63.69,42.41,25.86,0.73,0.76,0.68,0.73,Stars
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,199.0,349.0,0.43,4.0,43994.0,"Compatible with all Type C enabled devices, be...",https://m.media-amazon.com/images/W/WEBP_40237...,...,10.69,150.0,42.98,42.77,29.91,0.67,0.81,0.46,0.68,Mid-Tier
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,199.0,1899.0,0.90,3.9,7928.0,【 Fast Charger& Data Sync】-With built-in safet...,https://m.media-amazon.com/images/W/WEBP_40237...,...,8.98,1700.0,89.52,35.02,18.43,0.63,0.66,0.96,0.71,Mid-Tier
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,329.0,699.0,0.53,4.2,94363.0,The boAt Deuce USB 300 2 in 1 cable is compati...,https://m.media-amazon.com/images/I/41V5FtEWPk...,...,11.45,370.0,52.93,48.11,31.44,0.73,0.87,0.56,0.74,Stars
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,154.0,399.0,0.61,4.2,16905.0,[CHARGE & SYNC FUNCTION]- This cable comes wit...,https://m.media-amazon.com/images/W/WEBP_40237...,...,9.74,245.0,61.40,40.89,25.40,0.73,0.73,0.65,0.71,Stars


Review Preview:


,product_id,user_id,review_id,user_name,review_title,review_content
0,B07JW9H4J1,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...
1,B098NS6PVG,"AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...
2,B096MSW6CT,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a..."
3,B08HDJ86NZ,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou..."
4,B08CF3B7N1,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th..."


In [140]:
review_df["full_review"] = (
    review_df["review_title"].fillna("") + " " + review_df["review_content"].fillna("")
).str.strip()

In [141]:
def get_sentiment(text):
    if pd.isna(text) or text.strip() == " ":
        return np.nan
    return TextBlob(text).sentiment.polarity

In [142]:
review_df["sentiment_score"] = review_df["full_review"].apply(get_sentiment).round(2)
display(review_df[["full_review", "sentiment_score"]].head())

,full_review,sentiment_score
0,"Satisfied,Charging is really fast,Value for mo...",0.47
1,"A Good Braided Cable for Your Type C Device,Go...",0.29
2,"Good speed for earlier versions,Good Product,W...",0.50
3,"Good product,Good one,Nice,Really nice product...",0.33
4,"As good as original,Decent,Good one for second...",0.26


In [143]:
sentiment_summary = review_df.groupby("product_id").agg(
    avg_sentiment = ("sentiment_score", "mean"),
    text_review_count = ("full_review", "count")
).reset_index().round(2)

display(sentiment_summary.head())


,product_id,avg_sentiment,text_review_count
0,B002PD61Y4,0.35,2
1,B002SZEOLG,0.27,1
2,B003B00484,0.40,1
3,B003L62T7W,0.62,1
4,B004IO5BMQ,0.19,1


In [144]:
product_df = product_df.merge(sentiment_summary, on="product_id", how="left")


display(product_df[[
    "product_name",
    "rating",
    "rating_count",
    "avg_sentiment"
]].head())


,product_name,rating,rating_count,avg_sentiment
0,Wayona Nylon Braided USB to Lightning Fast Cha...,4.2,24269.0,0.47
1,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,4.0,43994.0,0.29
2,Sounce Fast Phone Charging Cable & Data Sync U...,3.9,7928.0,0.50
3,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,4.2,94363.0,0.33
4,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,4.2,16905.0,0.26


In [145]:
features2 = product_df[[
    "rating",
    "log_rating_count",
    "discount_percentage",
    "avg_sentiment"
]].fillna(0)

scaler2 = MinMaxScaler()
scaled2 = scaler2.fit_transform(features2)

product_df["rating_norm2"] = scaled2[:, 0].round(2)
product_df["log_rating_count_norm2"] = scaled2[:, 1].round(2)
product_df["discount_norm2"] = scaled2[:, 2].round(2)
product_df["sentiment_norm2"] = scaled2[:, 3].round(2)

In [146]:
product_df["enhanced_performance_score"] = (
    0.35 * product_df["rating_norm2"] +
    0.30 * product_df["log_rating_count_norm2"] +
    0.15 * product_df["discount_norm2"] +
    0.20 * product_df["sentiment_norm2"]
).round(2)

display(product_df[[
    "product_name",
    "rating",
    "rating_count",
    "avg_sentiment",
    "enhanced_performance_score"
]].sort_values("enhanced_performance_score", ascending=False).head(10))

,product_name,rating,rating_count,avg_sentiment,enhanced_performance_score
65,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...",4.4,426973.0,0.39,0.83
12,AmazonBasics Flexible Premium HDMI Cable (Blac...,4.4,426973.0,0.39,0.81
457,ELV Aluminum Adjustable Mobile Phone Foldable ...,4.5,28978.0,0.43,0.78
557,SanDisk Cruzer Blade 32GB USB Flash Drive,4.3,253105.0,0.45,0.78
551,Elv Aluminium Adjustable Mobile Phone Foldable...,4.5,28978.0,0.43,0.78
513,"SanDisk Ultra microSD UHS-I Card 64GB, 120MB/s R",4.4,67260.0,0.38,0.77
1029,Swiffer Instant Electric Water Heater Faucet T...,4.8,53803.0,0.59,0.77
40,AmazonBasics USB 2.0 Cable - A-Male to B-Male ...,4.5,107687.0,0.20,0.77
30,AmazonBasics USB 2.0 - A-Male to A-Female Exte...,4.5,74976.0,0.23,0.77
396,STRIFF PS2_01 Multi Angle Mobile/Tablet Tablet...,4.3,42641.0,0.35,0.76


In [147]:
review_df.to_csv("review_level_scored.csv", index=False)
product_df.to_csv("product_dashboard_dataset.csv", index=False)

print("Files saved:")
print("- review_level_scored.csv")
print("- product_dashboard_dataset.csv")

Files saved:
- review_level_scored.csv
- product_dashboard_dataset.csv
